1.没有记忆时

大模型是没有记忆的

In [8]:
from langchain.agents import create_agent

agent = create_agent(model = "deepseek-chat")

In [12]:
from langchain.messages import HumanMessage

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content = "你好，我是刚子，我最喜欢猫猫")]}  # 依旧犯了相同的错误，应该是messages 不是message
)

print(response)

{'messages': [HumanMessage(content='你好，我是刚子，我最喜欢猫猫', additional_kwargs={}, response_metadata={}, id='e2062e53-79f3-4ac4-9894-ae0ff469ec83'), AIMessage(content='你好，刚子！🐾  \n很高兴认识一位爱猫的朋友！猫猫真的是治愈系小天使，它们软乎乎的肉垫、高傲又黏人的性格，还有那些奇奇怪怪的可爱行为，总能让人瞬间心情变好～  \n\n你家里有养猫吗？还是云吸猫大军的一员呀？如果有猫，是哪位小主子在统治你家呀？🐱 如果是云吸猫，要不要聊聊你最喜欢的猫猫品种？或者分享一个你和猫猫的小故事？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 14, 'total_tokens': 117, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '31575c1d-81be-4339-92cc-d79361aeee1b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe6da-81cd-7173-ae16-8b789d15a9dc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_to

In [16]:
# 第二次调用
response = agent.invoke(
    {"messages": [HumanMessage(content = "我最喜欢什么动物？")]}
)

print(response)

{'messages': [HumanMessage(content='我最喜欢什么动物？', additional_kwargs={}, response_metadata={}, id='73a30ccb-990b-4fd5-abfc-caa8190895da'), AIMessage(content='哈哈，这个问题可难倒我啦！我猜不出你的心思，但可以给你几个「动物人格」选项对号入座：  \n\n- 如果你**爱撒娇又粘人**，可能喜欢 **狗**（比如金毛）；  \n- 如果你**独立又高冷**，可能悄悄爱着 **猫**；  \n- 如果你**向往自由**，或许会喜欢 **鸟** 或 **马**；  \n- 如果喜欢神秘感，可能迷上 **狐狸、蛇** 或 **章鱼**（对，章鱼是隐藏款！）；  \n- 甚至……你会不会其实最爱 **熊猫**，毕竟谁不爱圆滚滚呢～  \n\n偷偷告诉我，你更偏向哪种性格？或者，直接揭晓答案让我佩服一下？ 😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 9, 'total_tokens': 174, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '3e8181ba-6e0d-4a2a-b524-61b36d0c6151', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run-

大模型在“单次交互”中是无状态的，但在“同一会话”中可以通过工程手段拥有“伪记忆”。

记忆（Memory）分为两类：

短期记忆(short-term memory):当前任务或者会话的上下文

长期记忆(long-term memory)：跨任务或者会话的经验和知识

短期记忆(short-term memory)

在langchain中，短期记忆是通过**AgentState**实现的，而会话历史就是AgentState的一部分。

同时，langchain提供了**Checkpoint对象**来保存AgentState，每一次用户与AI交互都会生成一个快照，记录为一个checkpoint.

注意：同一个会话的多个checkpoint形成一个组，用同一个thread_id来标记。

To add short-term memory (thread-level persistence) to an agent, you need to specify a **checkpointer** when creating an agent.

In [17]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

In [19]:
agent = create_agent(
    "deepseek-chat",
    checkpointer = InMemorySaver()
)

In [20]:
from langchain.messages import HumanMessage

# 设定thread_id, 作为会话标识
config = {"configurable": {"thread_id": "thread_1"}}

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content = "你好，我叫刚哥，我喜欢猫猫")]},
    config   # 调用时添加thread_id
)

print(response)

{'messages': [HumanMessage(content='你好，我叫刚哥，我喜欢猫猫', additional_kwargs={}, response_metadata={}, id='6a43cf06-cb98-4523-a604-5f67bd18a00a'), AIMessage(content='你好，刚哥！很高兴认识你！🐱\n\n喜欢猫猫的人通常都很有爱心和耐心，相信你也是一个温柔的人。猫咪确实有种神奇的魔力，它们独立又黏人，高冷又撒娇，总能给生活带来很多乐趣和治愈感。\n\n不知道你家里有养猫吗？还是正在“云吸猫”阶段？如果有的话，可以分享一下你家主子的品种、名字或者趣事；如果还没养，也可以聊聊你最喜欢的猫咪品种，或者你梦想中的养猫生活。\n\n期待和你多聊聊关于猫猫的话题，也欢迎分享任何其他你感兴趣的事！😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 13, 'total_tokens': 140, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '5869b413-686f-4f8f-8fd3-5b97a5fa18b9', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe6ea-80a2-7503-9c94-232089a8945d-0', tool_calls=[],

In [21]:
# 第二次调用
response = agent.invoke(
    {"messages": [HumanMessage(content = "我最喜欢的动物是什么？")]},
    config  # 调用时添加thread_id
)

print(response)

{'messages': [HumanMessage(content='你好，我叫刚哥，我喜欢猫猫', additional_kwargs={}, response_metadata={}, id='6a43cf06-cb98-4523-a604-5f67bd18a00a'), AIMessage(content='你好，刚哥！很高兴认识你！🐱\n\n喜欢猫猫的人通常都很有爱心和耐心，相信你也是一个温柔的人。猫咪确实有种神奇的魔力，它们独立又黏人，高冷又撒娇，总能给生活带来很多乐趣和治愈感。\n\n不知道你家里有养猫吗？还是正在“云吸猫”阶段？如果有的话，可以分享一下你家主子的品种、名字或者趣事；如果还没养，也可以聊聊你最喜欢的猫咪品种，或者你梦想中的养猫生活。\n\n期待和你多聊聊关于猫猫的话题，也欢迎分享任何其他你感兴趣的事！😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 13, 'total_tokens': 140, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '5869b413-686f-4f8f-8fd3-5b97a5fa18b9', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe6ea-80a2-7503-9c94-232089a8945d-0', tool_calls=[],